# Frozen speech features: one-recording check

Özgür Ekin Sönmez

Measure the fixed wav2vec 2.0 encoder on one public train recording before extracting the pilot. No labels, development audio, fine-tuning or model comparison is performed here. Use a free GPU runtime; do not upgrade to a paid plan. The uploaded bundle reuses verified local weights, so no second checkpoint download is needed.


In [ ]:
import json, os, shutil, subprocess, sys
from pathlib import Path
import torch
hardware = {'python': sys.version, 'torch': torch.__version__, 'cuda': torch.cuda.is_available(),
            'disk_free_bytes': shutil.disk_usage('/content').free,
            'memory': Path('/proc/meminfo').read_text()}
if torch.cuda.is_available():
    hardware['gpu_name'] = torch.cuda.get_device_name(0)
    hardware['gpu_free_total_bytes'] = list(torch.cuda.mem_get_info())
Path('/content/ssl_hardware.json').write_text(json.dumps(hardware, indent=2))
print(json.dumps(hardware, indent=2))
assert torch.cuda.is_available(), 'Choose Runtime > Change runtime type > T4 GPU, then reconnect.'
assert tuple(map(int, torch.__version__.split('+')[0].split('.')[:2])) >= (2,10), 'Patched PyTorch >=2.10 is required. Stop and review this runtime; do not install an unplanned GPU stack.'
assert hardware['disk_free_bytes'] >= 20 * 1024**3

## Lock the optional packages

Keep the runtime-provided patched GPU PyTorch. Resolve and hash-lock the small encoder packages before installing them. The Linux lock and full environment are saved with the result. Restart the runtime if an already imported package conflicts; do not change the feature or model settings.


In [ ]:
import hashlib, importlib.metadata as metadata
pins = ['transformers==4.57.6','numpy==2.2.6','soundfile==0.13.1',
        'huggingface-hub==0.36.2','tokenizers==0.22.2','safetensors==0.8.0']
subprocess.run([sys.executable,'-m','pip','install','--dry-run','--ignore-installed',
                '--only-binary=:all:','--report','/content/ssl_resolve.json',*pins], check=True)
resolution = json.loads(Path('/content/ssl_resolve.json').read_text())
assert not any(item['metadata']['name'].lower() == 'torch' for item in resolution['install'])
lock = []
for item in resolution['install']:
    download = item['download_info']
    lock.append(download['url']+' --hash=sha256:'+download['archive_info']['hashes']['sha256'])
Path('/content/ssl-linux.lock').write_text('\n'.join(lock)+'\n')
subprocess.run([sys.executable,'-m','pip','install','--no-deps','--require-hashes',
                '-r','/content/ssl-linux.lock'], check=True)
subprocess.run([sys.executable,'-m','pip','check'], check=True)
versions = {item.metadata['Name']:item.version for item in metadata.distributions()}
Path('/content/ssl_environment.json').write_text(json.dumps(versions, indent=2))
print('Encoder packages are locked. Existing unrelated Colab package conflicts must be reviewed before the smoke.')

## Upload the local smoke bundle

Select `ssl_colab_smoke_v1.zip` from the project outputs folder. It contains the fixed public training recording, verified model files, geometry code, the measured CPU feature reference and the bounded smoke runner. No private recordings, labels or development audio are included.


In [ ]:
from google.colab import files
uploaded = files.upload()
assert len(uploaded) == 1 and 'ssl_colab_smoke_v1.zip' in uploaded
import zipfile
bundle = Path('/content/ssl-smoke-bundle')
assert not bundle.exists(), 'Keep previous attempts. Use a fresh runtime for another smoke.'
bundle.mkdir()
with zipfile.ZipFile('/content/ssl_colab_smoke_v1.zip') as archive:
    assert sum(item.file_size for item in archive.infolist()) < 500_000_000
    for item in archive.infolist():
        target = (bundle / item.filename).resolve()
        assert target.is_relative_to(bundle.resolve()), 'Unsafe archive path'
    archive.extractall(bundle)
print('Bundle extracted. The worker will verify every pinned payload before loading.')

## Run the bounded GPU check

The parent watches system memory and the worker process for 180 seconds. The worker records GPU memory and compares GPU features with the preliminary CPU reference; numerical differences are reported, not tuned away. A stopped attempt is kept and must be reviewed. This notebook does not launch the 192-recording extraction automatically.


In [ ]:
command = [sys.executable, str(bundle/'ssl_colab_smoke.py'),'--execute',
           '--bundle-dir', str(bundle),'--output-dir','/content/ssl-colab-smoke-v1']
completed = subprocess.run(command, timeout=200)
assert completed.returncode == 0, 'Smoke stopped: inspect status and worker logs; do not automatically retry.'
print(Path('/content/ssl-colab-smoke-v1/status.json').read_text())
print(Path('/content/ssl-colab-smoke-v1/result.json').read_text())

In [ ]:
import zipfile
result_zip = Path('/content/ssl_colab_smoke_result_v1.zip')
with zipfile.ZipFile(result_zip,'x',compression=zipfile.ZIP_DEFLATED) as archive:
    for path in Path('/content/ssl-colab-smoke-v1').rglob('*'):
        if path.is_file(): archive.write(path, str(path.relative_to('/content')))
    for name in ['ssl_hardware.json','ssl_resolve.json','ssl-linux.lock','ssl_environment.json']:
        archive.write('/content/'+name,name)
files.download(str(result_zip))